# 03 — Cross-evaluation: every trained model on every validation set

**Work package:** WP7 (validation) · **Runtime: T4 GPU preferred** (~5 min); CPU works (~20 min).
No training happens here.

**Why:** run 1 (Roboflow split) and run 2 (blocked split) were each scored on *their own*
validation set. Those sets differ — different frames, different class mix (e.g. 34 vs 117
packaging-error boxes) — so their scores cannot be compared directly (RESULTS_LOG R3).
A fair test of leakage scores **both models on the same images**:

| Model | Trained on | On the **blocked VALID** images it has… |
|---|---|---|
| `yolo11n_640` (run 1) | Roboflow TRAIN | seen neighbouring frames of the same trays (**leaky**) |
| `yolo11n_640_blocked` (run 2) | blocked TRAIN | never seen those trays (**clean**) |

Same architecture, same settings, same evaluation images → the score difference is the
benefit of having seen the trays: the **optimism caused by leakage**. Models already on
Drive are evaluated; missing ones are skipped. The TEST split is still not touched (D-008).

## 1. Settings

In [ ]:
RUN_NAMES = ["yolo11n_640", "yolo11n_640_blocked", "yolo26n_640_blocked"]  # missing runs are skipped
EVAL_SETS = ["roboflow_valid", "blocked_valid"]
PUSH_RESULTS = True

## 2. Code, environment, Drive

In [ ]:
import os, sys, json, subprocess, datetime, platform
REPO_URL = "https://github.com/Hudson11111/MMA3001Project.git"
REPO_DIR = "/content/MMA3001Project"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q main && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

import torch, ultralytics, porkqc
from pathlib import Path
from google.colab import drive
from porkqc.config import load_yaml
env_info = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "torch": torch.__version__, "ultralytics": ultralytics.__version__,
    "git_commit": subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(env_info, indent=2))
if not torch.cuda.is_available():
    print("NOTE: no GPU - predictions will run on CPU (slower, ~20 min). Timings from this run are CPU timings.")
drive.mount("/content/drive")
cfg = load_yaml("configs/dataset.yaml")
RUNS_DIR = Path(cfg["paths"]["drive_project_dir"]) / "runs"
!ls {RUNS_DIR}

## 3. Datasets: Roboflow split and time-blocked split

In [ ]:
from porkqc import CLASS_NAMES
from porkqc.data.download import download_dataset, fix_data_yaml
from porkqc.data.resplit import collect_frames, assign_time_blocks, materialise_split

rf = cfg["roboflow"]
data_root = Path(cfg["paths"]["colab_data_dir"])
if not (data_root / "data.yaml").exists():
    data_root = download_dataset(rf["workspace"], rf["project"], rf["version"], data_root, export_format=rf["export_format"])
fix_data_yaml(data_root / "data.yaml")
b = cfg["blocked_split"]
blocked_root = Path(b["colab_dir"])
frames_map, files_map, _ = collect_frames(data_root)
assign = assign_time_blocks(frames_map, b["block_size"], b["fractions"], b["gap"], b["seed"])
print("blocked split:", materialise_split(data_root, blocked_root, assign, CLASS_NAMES, files=files_map))
SET_DIRS = {"roboflow_valid": data_root / "valid", "blocked_valid": blocked_root / "valid"}

## 4. Predict (skips files that already exist)

Same prediction settings as training notebook 02 (`configs/experiments/*.yaml`).

In [ ]:
from porkqc.models.detector import load_experiment, predict_images
from porkqc.predictions import export_ground_truth, save_json, load_json
from porkqc.data.audit import list_images

PRED_DIR, GT_DIR = Path("results/predictions"), Path("results/ground_truth")
for s in EVAL_SETS:
    gt_path = GT_DIR / f"{s}.json"
    if not gt_path.exists():
        save_json(export_ground_truth(SET_DIRS[s], CLASS_NAMES, split=s), gt_path)

made = []
for run in RUN_NAMES:
    weights = RUNS_DIR / run / "weights" / "best.pt"
    if not weights.exists():
        print(f"skip {run}: no weights on Drive"); continue
    exp = load_experiment(f"configs/experiments/{run.replace('_blocked', '')}.yaml")
    for s in EVAL_SETS:
        for variant in exp["predict_variants"]:
            out = PRED_DIR / f"xeval_{run}_on_{s}_{variant['tag']}.json"
            if out.exists():
                continue
            kw = {**exp["predict"], **{k: v for k, v in variant.items() if k != "tag"}}
            pred = predict_images(weights, list_images(SET_DIRS[s] / "images"), run, s, **kw)
            pred["git_commit"] = env_info["git_commit"]
            save_json(pred, out, indent=None); made.append(out)
            print("wrote", out)

## 5. Score every model on every set (our tested metrics, exact AP)

Confidence intervals: bootstrap resampling of **50-frame runs** (not single images), because
neighbouring frames within a validation set are not independent.

In [ ]:
import numpy as np, pandas as pd
from porkqc.data.audit import frame_number
from porkqc.metrics.evaluate import match_dataset, summarise, bootstrap

rows = []
for f in sorted(PRED_DIR.glob("xeval_*_default.json")):
    pred = load_json(f); s = pred["split"]
    m = match_dataset(load_json(GT_DIR / f"{s}.json"), pred, len(CLASS_NAMES))
    for x in m:
        x.group = str(frame_number(x.file) // 50)
    summ = summarise(m, CLASS_NAMES)
    ci = bootstrap(m, lambda mm: summarise(mm, CLASS_NAMES)["map50"], n_boot=500, seed=0, grouped=True)
    pc = {c["class"]: c["ap50"] for c in summ["per_class"]}
    rows.append({"model": pred["model"], "eval set": s, "mAP50": summ["map50"], "CI low": ci["low"], "CI high": ci["high"],
                 "mAP50-95": summ["map50_95"], "AP50 unsealed": pc["unsealed"], "AP50 packaging-error": pc["packaging-error"]})
table = pd.DataFrame(rows).round(3)
display(table)
Path("results/xeval").mkdir(parents=True, exist_ok=True)
table.to_csv("results/xeval/cross_evaluation.csv", index=False)

def get(model, s):
    r = table[(table.model == model) & (table["eval set"] == s)]
    return None if r.empty else float(r["mAP50"].iloc[0])
leaky, clean = get("yolo11n_640", "blocked_valid"), get("yolo11n_640_blocked", "blocked_valid")
if leaky is not None and clean is not None:
    print(f"Leakage optimism on blocked VALID: leaky model {leaky:.3f} vs clean model {clean:.3f} "
          f"-> difference {leaky - clean:+.3f} mAP50")

## 6. Save / push

In [ ]:
import shutil
from porkqc.env import get_secret
from porkqc.colab_utils import push_results
save_json(env_info, Path("results/xeval/notebook_env.json"))
paths = [Path("results/xeval"), *made, *[GT_DIR / f"{s}.json" for s in EVAL_SETS]]
try:
    if not PUSH_RESULTS:
        raise RuntimeError("PUSH_RESULTS is False")
    sha = push_results(REPO_DIR, [str(p) for p in paths], "results: cross-evaluation (Colab)", get_secret("GITHUB_TOKEN"))
    print("Pushed to 'colab-results':", sha[:7])
except (RuntimeError, ValueError) as exc:
    print("Not pushed:", exc)
    archive = shutil.make_archive("/content/results_xeval", "zip", ".", "results")
    from google.colab import files
    files.download(archive)

Send Claude the zip (or say "pushed"), then *Runtime → Disconnect and delete runtime*.